# diabetes: classification (batch, databricks)

**Use case:** `usecase-type/batch/classification/databricks-notebook` — bounded scoring of many requests into stored predictions with the Databricks job with shared notebooks.

**Training:** Databricks job task `train-evaluate` running the shared accelerator notebook on an existing cluster.

**Serving:** Databricks task `batch-score`: Spark scoring of a Unity Catalog table into a Delta table.

**Status:** Executable notebook templates over shared engines; cloud steps are explicit, charged operations behind switches that default to false and need the listed prerequisites.

Kaggle-only source: `jamaltariqcheema/pima-indians-diabetes-dataset`.
Dataset status: **configured**.
Teaching-only classification, not clinical decision support. Historical cohort is not representative of all populations. Physiologically implausible zero values require a reviewed domain-specific missingness policy before production.

**Prerequisites**

- Review the scenario's dataset status, license and notes; gated datasets stay blocked until approved.
- An existing Databricks workspace and cluster with the shared notebooks and factory wheel imported; Unity Catalog objects and permissions per `user-config/databricks`.

**Limitations**

- Databricks models live in Databricks MLflow/Unity Catalog, not in the Azure ML registry.
- Templates are not evidence of a completed Kaggle, Azure or Databricks run.

This notebook is an executable template, not evidence of a completed Kaggle or Azure run.
Install the factory's declared extras into this kernel separately. No packages,
credentials, cloud resources, or competition terms are installed/accepted by this notebook.
All execution switches default to false. Run **either** SDK v2 **or** CLI v2 submission, not both.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "accelerator" / "src" / "ml_model_factory" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the 50-ml-model-factory checkout")
SOURCE = ROOT / "accelerator" / "src"
sys.path.insert(0, str(SOURCE))
from ml_model_factory.config import load_json, validate_scenario, validate_runtime, write_json

SCENARIO = ROOT / "user-config" / "model" / "scenarios" / "diabetes.json"
scenario = validate_scenario(load_json(SCENARIO))
MODE = "custom"
PATTERN = "batch"
TECHNOLOGY = "databricks-notebook"
RUNTIME = Path("user-config/runtime.local.json")
if not RUNTIME.is_absolute():
    RUNTIME = ROOT / RUNTIME
WORK = ROOT / "ml-environment" / "outputs" / scenario["name"] / "databricks"
RAW = ROOT / "data" / "in" / scenario["name"]
PREPARED = ROOT / "data" / "out" / scenario["name"] / "databricks" / "prepared"
MODEL = WORK / "model"
EVALUATION = WORK / "evaluation"
RENDERED = WORK / "azureml"
INFERENCE = WORK / PATTERN
REQUESTS = INFERENCE / "requests"
MODEL_NAME = scenario.get("model_name", scenario["name"])
MODEL_ID = ""  # immutable azureml:<name>:<version> after quality-gated registration
RUN_INGEST = False
RUN_PREPARE = False
SAMPLE_REQUESTS = False
RENDER_DATABRICKS_JOB = False
CREATE_DATABRICKS_JOB = False
RUN_DATABRICKS_JOB = False
DATABRICKS_HOST = ""  # https://adb-<workspace-id>.<n>.azuredatabricks.net
DATABRICKS_SCENARIO_PATH = "/Workspace/Shared/ml-model-factory/user-config/model/scenarios/" + scenario["name"] + ".json"
DATABRICKS_JOB_ID = ""
LAKE_CONFIG = ROOT / "user-config" / "lake.local.json"
LAKE_ROOT = ROOT / "data" / "out" / "lake-data"
EFFECTIVE_RUNTIME = RUNTIME
FACTORY_ENV = os.environ.copy()
FACTORY_ENV["PYTHONPATH"] = str(SOURCE) + os.pathsep + FACTORY_ENV.get("PYTHONPATH", "")
FACTORY_ENV.setdefault("MLFLOW_TRACKING_URI", (ROOT / "ml-environment" / "mlruns").as_uri())

def factory(*arguments):
    command = [sys.executable, "-m", "ml_model_factory", *map(str, arguments)]
    subprocess.run(command, cwd=ROOT, env=FACTORY_ENV, check=True)


## Kaggle ingestion and reproducible preparation

Review the source license and any competition rules first. Required-selection and
license-review gates block ingestion. Do not clear them without documented approval.
The download manifest records file hashes and pinned dataset version. Existing nonempty
outputs are never silently replaced. Titanic preparation additionally writes `titanic.parquet`.
Forecasting uses chronological holdouts, not random rows.


In [ ]:
if RUN_INGEST:
    factory("ingest", "--scenario", SCENARIO, "--output", RAW)
INPUT = RAW / scenario["dataset"]["file"]
if RUN_PREPARE:
    factory("prepare", "--scenario", SCENARIO, "--input", INPUT, "--output", PREPARED)
if (PREPARED / "manifest.json").exists():
    print(json.dumps(load_json(PREPARED / "manifest.json"), indent=2))


## Unlabeled requests from the held-out split

`sample-requests` writes demonstration requests without labels; `requests.parquet` is the unlabeled batch input.
`labels.parquet` keeps held-out truth separately for monitoring demos, and forecasting
also writes `history.parquet` for AutoML scoring. Never send labels to a model.


In [ ]:
if SAMPLE_REQUESTS:
    factory("sample-requests", "--scenario", SCENARIO, "--prepared", PREPARED, "--output", REQUESTS, "--rows", 10)
if (REQUESTS / "manifest.json").exists():
    print(json.dumps(load_json(REQUESTS / "manifest.json"), indent=2))


## Databricks job: `train-evaluate` then `batch-score`

`databricks-job render` combines the customer job template (existing cluster, wheel and
notebook locations) with `inference-settings` for this pattern and the shared accelerator
notebooks; the model URI flows between tasks as a task value. Rendering reports unresolved
placeholders; creating or running the job refuses them. Authentication uses your Azure CLI
sign-in (or managed identity in automation), never personal access tokens.


In [ ]:
DATABRICKS_TEMPLATE = ROOT / "user-config" / "databricks" / "job-template.json"
DATABRICKS_SETTINGS = ROOT / "user-config" / "databricks" / "inference-settings.local.json"
DATABRICKS_JOB = INFERENCE / "databricks-job.json"
if RENDER_DATABRICKS_JOB:
    factory("databricks-job", "render", "--scenario", SCENARIO, "--pattern", PATTERN,
            "--template", DATABRICKS_TEMPLATE, "--settings", DATABRICKS_SETTINGS,
            "--scenario-path", DATABRICKS_SCENARIO_PATH, "--orchestration", "databricks",
            "--output", DATABRICKS_JOB)
if CREATE_DATABRICKS_JOB:
    factory("databricks-job", "create", "--job", DATABRICKS_JOB, "--host", DATABRICKS_HOST,
            "--auth", "azure-cli", "--execute")
if RUN_DATABRICKS_JOB:
    if not DATABRICKS_JOB_ID:
        raise ValueError("Set DATABRICKS_JOB_ID to the created job")
    # The receipt reuses the idempotency token on retry and refuses a second run while one is unresolved.
    factory("databricks-job", "run", "--job-id", DATABRICKS_JOB_ID, "--host", DATABRICKS_HOST,
            "--auth", "azure-cli", "--wait-for", "terminated", "--receipt", INFERENCE / "databricks-run.json",
            "--execute")


## Monitoring and clean-up

Join predictions with observed labels by `request_id` only after outcomes are known, then use
`python -m ml_model_factory monitor` with explicit reference/current windows (see the
model-factory guide). Predictions are never labels. Batch compute scales back to zero; storage of inputs and predictions remains.
